# clipsmith — Twitch VOD → AI Vertical Clips (Colab T4)

**What this notebook does:**
1. Installs clipsmith and GPU transcription dependencies
2. Mounts your Google Drive for persistent storage
3. Downloads a Twitch VOD, transcribes it (T4 GPU, ~10 min for 3 hrs)
4. Uses Claude (Haiku) to select the best 15–30 s clip moments
5. Cuts final clips with ffmpeg and lets you preview them inline

**Requirements:**
- Colab runtime: **GPU → T4** (Runtime → Change runtime type)
- Add your `ANTHROPIC_API_KEY` to Colab Secrets (🔑 left panel)
- Optional: add `TWITCH_CLIENT_ID` and `TWITCH_CLIENT_SECRET` to fetch existing clip data

Each cell is re-runnable — cached transcripts and picks are loaded from disk on subsequent runs.

In [ ]:
# @title 1. Install clipsmith and GPU dependencies
# Takes ~2 min on first run; skip on subsequent runs if kernel is still alive.
!pip install -q "git+https://github.com/ricardogr07/clipsmith.git"
# GPU-capable faster-whisper (CUDA 11.x, bundled with T4 Colab runtime)
!pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11
# ffmpeg is pre-installed on Colab; verify:
!ffmpeg -version | head -1

In [ ]:
# @title 2. Mount Google Drive and load API keys from Colab Secrets
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# Optional Twitch credentials (for fetching existing clip boosts):
# os.environ['TWITCH_CLIENT_ID']     = userdata.get('TWITCH_CLIENT_ID')
# os.environ['TWITCH_CLIENT_SECRET'] = userdata.get('TWITCH_CLIENT_SECRET')
print("Drive mounted. API key loaded.")

In [ ]:
# @title 3. Configure — set your VOD ID here
from pathlib import Path

VIDEO_ID = "2758856582"  # <-- paste your Twitch VOD ID
WORK_DIR = "/content/drive/MyDrive/clipsmith/work"
OUT_DIR = "/content/drive/MyDrive/clipsmith/out"

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

config_yaml = f"""\
work_dir: {WORK_DIR}
out_dir:  {OUT_DIR}

transcribe:
  model: medium          # medium balances speed/quality; large-v3 for best accuracy
  compute_type: float16  # float16 = full T4 GPU precision
  language: es           # change to 'en' for English streams

llm:
  provider: anthropic
  model_anthropic: claude-haiku-4-5-20251001  # cheapest; use claude-sonnet-4-6 for more quality

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none  # set to 'center' for 9:16 vertical crop

caption:
  enabled: false
"""

Path("config.yaml").write_text(config_yaml)
print(f"Config written. VOD: {VIDEO_ID}")

In [ ]:
# @title 4. Download VOD and transcribe (GPU)
# Transcription of a 3-hour stream takes ~10 min on T4 vs ~3 hrs on CPU.
# Re-running this cell skips both steps if cached files already exist.
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-select",
        "--skip-clip",
    ],
    check=True,
)
print("Download + transcription complete.")

In [ ]:
# @title 5. LLM clip selection (Claude Haiku)
# Picks the best 15–30 s moments from the transcript using your API key.
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-download",
        "--skip-transcribe",
        "--skip-chat",
        "--skip-clip",
    ],
    check=True,
)
print("Clip selection complete.")

In [ ]:
# @title 6. Cut final clips with ffmpeg
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-download",
        "--skip-transcribe",
        "--skip-chat",
        "--skip-select",
    ],
    check=True,
)
print("Clips cut and saved to Drive.")

In [ ]:
# @title 7. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clip_dir = Path(OUT_DIR) / VIDEO_ID
clips = sorted(clip_dir.glob("clip_*.mp4"))

if not clips:
    print("No clips found. Did step 6 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))